In [ ]:
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )

include( "../0-params/nos-list.jl" )

using Optim
using ThreadsX

home, base = expfolder( "system" )

colormap = cgrad( :ice; rev=true )

In [ ]:
# Hyper parameters.
N = 1024;  ρ = 0.251;  s = 0.251

# Create folder and extract critical noise.
folder = createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) )

# Extract cluster mass distribution exponent.
λ = readdlm( folder*"power-law.txt" )[1]

# Import the cluster change frequency values.
θlist = readdlm( folder*"optimal-rates.txt" )

# Extract the indeces with data points.
# Note: Also enforcing upper bound.
ιlist = BitVector( [
    all( [isfinite( θlist[j,k] ) & (θlist[j,k] < 5.0) &
        (k < 3 ? 1.0 < θlist[j,k] : true) for k ∈ 1:3] )
    for j ∈ 1:N] );

In [ ]:
# Left and right distributions expectation.
λ1 = mean( θlist[ιlist,1] )
λ2 = mean( θlist[ιlist,2] )
κ  = mean( θlist[ιlist,3] );

In [ ]:
i = 15

# Plot the optimal rates for changing cluster size.
plt = plot(; layout=(3,1), size=(400,600), legend=false )

for j ∈ 1:2
    # Exponent for changing cluster size.
    scatter!( plt[j], 1:N, θlist[:,j]; color=:gray )

    # Exponent from the cluster mass distribution.
    plot!( plt[j], [1,N], fill( λ, 2 ); color=:indianred, linestyle=:dash, lw=2 )

    # Plot details.
    plot!( plt[j]; xlims=(1,N), ylims=(0,3) )
end

scatter!( plt[3], 1:N, θlist[:,3]; color=:gray )
plot!( plt[3]; xlims=(1,N), ylims=(0,1) )

plot!( plt[1], [1,N], fill( λ1, 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )
plot!( plt[2], [1,N], fill( λ2, 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )
plot!( plt[3], [1,N], fill(  κ, 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )

In [ ]:
# Compile the approximated rate variables.
θ̂list = [fill( λ1, N ) fill( λ2, N ) fill( κ, N )];

In [ ]:
# Save parameter data.
if true
    # Save the cluster change parameters for each parameter case.
    writedlm( folder*"optimal-rates-hat.txt", θ̂list )
end